> ## SOLUTION / ANSWER KEY &mdash; Lab 3.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-mini-transformer.ipynb`](../lab-12-capstone-mini-transformer.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 3.12 &mdash; Capstone: A Real Sentence-Encoder Pipeline

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Chain tokenize -> real model -> mean-pool -> sentence vector
- Produce one vector that represents a whole sentence
- Use those vectors for real semantic matching

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; The transformer block](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-12")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Everything in Module 3 now snaps together on a **real** model: **tokenize** a sentence, run it
through the transformer to get **contextual token vectors** (`last_hidden_state`), **mean-pool** them
into one **sentence vector**, and use those vectors for **semantic matching**. This is exactly what a
real sentence-encoder / retrieval system does &mdash; you are building it from the parts you learned.

## Build it
Complete the pooling step and the nearest-match selection.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel
_M = {}
def _load():
    if not _M:
        name = "sentence-transformers/all-MiniLM-L6-v2"
        _M['tok'] = AutoTokenizer.from_pretrained(name)
        _M['mdl'] = AutoModel.from_pretrained(name); _M['mdl'].eval()
    return _M['tok'], _M['mdl']

def encode(sentence):
    tok, mdl = _load()
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad(): out = mdl(**enc)
    H = out.last_hidden_state[0]                 # (tokens, dim) real contextual vectors
    v = H.mean(dim=0)
    v = v / v.norm()                             # unit vector
    return v.numpy()

def best_match(query, corpus):
    qv = encode(query)
    sims = [float(np.dot(qv, encode(s))) for s in corpus]
    i = int(np.argmax(sims))
    return corpus[i], round(sims[i], 3)

## Run it for real
Run the full real pipeline and match queries to a small corpus.

In [ ]:
try:
    # see the tokenize step for one sentence, then use the whole pipeline
    tok, _ = _load()
    print("tokens for 'a small furry pet':", tok.tokenize("a small furry pet"))

    CORPUS = ["cats and kittens are small furry pets",
              "the stock market fell sharply today",
              "transformers use attention to understand language",
              "he cooked a delicious italian pasta dinner"]
    for query in ["a cute little kitten", "how do neural networks read text", "what should i eat tonight"]:
        match, score = best_match(query, CORPUS)
        print(f"\nQUERY: {query}\n  best -> {match}  (cos={score})")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- One real pipeline: **tokenize -> model -> mean-pool -> unit vector**, every stage from Module 3 in order.
- Each query lands on the **right** corpus sentence by meaning &mdash; with no shared keywords.
- `last_hidden_state` holds **contextual** token vectors (a word's vector depends on its neighbours); mean-pooling collapses them to a fixed-size sentence vector a classifier or index can use.

## Your turn (open task &mdash; no grader)
Extend the corpus, add paraphrase queries, or swap the model for `distilbert-base-uncased`
inside `_load()` &mdash; do the matches change? Then try **max-pooling** or **CLS-token** pooling
(`H[0]`) instead of mean and compare. A "good" answer: you can name each stage of your pipeline and
say what would break if you removed it.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    CORPUS2 = CORPUS + ["the puppy chased a ball across the park",
                        "she solved the equation on the whiteboard"]
    for query in ["a tiny meowing feline", "teaching a computer to read", "outdoor play with a pet"]:
        match, score = best_match(query, CORPUS2)      # paraphrase queries, extended corpus
        print(f"QUERY: {query}\n  best -> {match}  (cos={score})")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
You built a real sentence encoder from the parts you learned -- tokenize, embed, attend, pool. Module 4 takes real pretrained models and fine-tunes them for specific tasks. That completes Module 3.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>